# 第62课 · 教会 PyTorch「吃」声音——Dataset / DataLoader 把一秒语音喂成批量 Mel 特征

**目标**：Dataset / DataLoader 把一秒语音喂成批量 Mel 特征（可离线合成回退）。

> 全程用自研 `aurora.audio.mel`，不调用 `torchaudio.transforms`。路径 / shape 失败先查工作目录与划分理由。

🔗 Aurora 连接：`aurora.audio.mel.mel_spectrogram()` — 本节全程使用自己实现的版本，绝不调用 `torchaudio.transforms`。

### 为什么用 Mel 频谱？——人耳不是"线性感知"

想象你在钢琴上按下两个键：

- **低音区**：100 Hz 和 200 Hz，间隔 100 Hz。你的耳朵能清楚区分吗？**能**。
- **高音区**：10000 Hz 和 10100 Hz，间隔也是 100 Hz。你能听出差别吗？**几乎听不出**。

为什么？**因为人耳对频率的分辨率是*相对*的（百分比），不是绝对的**。

```
人耳的"距离感" ∝ log(频率)
```

这个观察被量化为 **Mel 标度**（Mel = melody）：

| 频率 (Hz) | Mel 值 | 直观含义 |
|---|---|---|
| 0 | 0 | 无声 |
| 100 | ≈ 150 | 低音 A |
| 500 | ≈ 632 | 中低音 |
| 1000 | ≈ 1000 | 中音 |
| 2000 | ≈ 1870 | 中高音 |
| 10000 | ≈ 2920 | 高音，但 Mel 增长变缓 |

**公式**：$\text{Mel} = 2595 \times \log_{10}\left(1 + \frac{f}{700}\right)$

你看出什么了吗？**Mel 值增长越来越慢** — 这就是对数的特点。

**在 Mel 频谱中，一个 Mel bin 代表人耳感知的"等距离"频率间隔**。低频密集，高频稀疏，完全符合人类听觉的不均匀性。

**为什么 CNN 喜欢 40 个 Mel bin？**

- 513 个线性 FFT bin：高频纯噪声、冗余巨大，CNN 需要更多参数区分无意义的细节 → 过拟合风险。
- 40 个 Mel bin：按人耳感知重新组织，信号与噪声的对比被凸显，分类器学得更快。

这节课里，**`aurora.audio.mel.mel_spectrogram()` 内部做的就是**：
1. STFT → 513 个线性频点
2. 通过 Mel 滤波器组（40 个重叠的三角形滤波器，中心按 Mel 标度排列）
3. 把 513 维压缩到 40 维，同时保留人耳关心的信息


← **上一课**　[L61 · nn.Module 实战](L61_pytorch_nn.ipynb)

> 上节课学习了 **nn.Module 实战**：Linear / ReLU / Sequential，参数管理与模型保存。  
> 本课将探讨 **Dataset 与 DataLoader**。

## 本课剧情：把"一秒钟的声音"压缩成一张地图

语音识别的第一个难题：神经网络吃的是固定长度的数字，但人说"yes"可能花 0.8 秒，说"no"可能花 0.6 秒，波形长短不一，怎么办？

**两步预处理**，让声音变成标准地图：

```
原始波形 (sr=16000, 不定长)
   ↓ pad/truncate → 正好 16000 个采样点 (1秒)
   ↓ STFT + Mel 滤波器组 → (n_mels=40, T≈32帧) 的"时频图"
   ↓ normalize → 均值=0，方差=1
   ↓ 送进 CNN/MLP
```

**为什么用 Mel 频谱而不是原始波形**：

| 表示 | 维度 | 包含信息 | CNN 是否适用 |
|---|---|---|---|
| 原始波形 | 16000 点 | 全部 | 勉强，但冗余大 |
| FFT 线性频谱 | 513 频点 × T | 全频率 | 高频细节多余 |
| Mel 频谱 | 40 × T | 人耳感知频率 | ✅ 最佳 |

**Speech Commands 数据集**：35 类关键词（yes/no/up/down…），每条约 1 秒，16kHz，~2.3GB。

本节任务：实现 `extract_features(wav_path)` — 读取 WAV → pad/truncate → Mel → dB → 归一化（normalization） → 返回 `(40, T)` Tensor。

In [1]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [2]:
import torch
import numpy as np
from aurora.audio.io import read_wav, write_wav
from aurora.audio.mel import mel_spectrogram, power_to_db
import matplotlib.pyplot as plt
%matplotlib inline

# 依赖说明：本课必需 torch（安装：pip install -e ".[speech]"）；
# torchaudio 只在使用真实 Speech Commands 数据时才需要——
# 缺了它（或没有网络）本课会自动改用合成关键词数据，照样能完整跑通。

## 1. Speech Commands 数据集（离线自动回退到合成数据）

Speech Commands 包含 35 类英文关键词（yes / no / up / down 等），每段约 **1 秒**，采样率（sample rate，SR） **16 kHz**，单声道 WAV 文件。标签用子目录名表示，每个文件形如：
```
data/SpeechCommands/speech_commands_v0.02/yes/0a7c2a8d_nohash_0.wav
```
类别数 35，每类约 2000–4000 条，总计约 105 000 条训练样本。

> **下载体量提示**：完整归档约 **2.3 GB**。本课默认**不**自动下载——下方代码只在检测到 torchaudio 且本地已有数据时才用真实数据；否则自动回退到**固定种子的合成关键词数据集**（不同关键词用不同频带的扫频音模拟），保证无网络、无 torchaudio 也能完整跑通全课。
>
> 想用真实数据：`pip install -e ".[speech]"` 安装 torchaudio 后，把下方 `download=False` 改成 `True` 再运行一次。

两条路径产出的样本格式完全一致：`(waveform_tensor(1, N), sr, label, ...)`，后面所有 cell 都不用管数据是从哪来的。

#### 采样率 16 kHz 到底是什么意思？

**16 kHz = 16 000 个样本 / 秒**

所以如果你要记录**恰好 1 秒**的音频，需要采集：
$$\text{总采样数} = \text{采样率} \times \text{时长} = 16000 \, \text{样本/秒} \times 1 \, \text{秒} = 16000 \, \text{样本}$$

这是高中数学的"单位转换"——想象买 16 元/斤的猪肉，买 1 斤就是 16 元，这里买 1 秒的声音，采样率 16000 就得到 16000 个数字。

反过来，如果你有一个长度未知的 WAV 文件，想知道它播放多久：
$$\text{播放时长（秒）} = \frac{\text{采样数}}{\text{采样率}} = \frac{\text{len}(x)}{16000}$$


In [3]:
# 数据加载：优先真实 Speech Commands，失败则回退到合成关键词数据集
SR = 16000

def _synth_keyword(label: str, rng: np.random.Generator) -> np.ndarray:
    """合成 1 秒关键词波形：不同标签 = 不同频带的扫频音 + 高斯包络 + 轻噪声。
    
    三个关键概念：
    
    1. 扫频音（Chirp）：频率不固定，而是从 A Hz 线性扫到 B Hz。
       - "yes" 用高频扫频（2000→3500 Hz），"no" 用低频扫频（200→350 Hz）
       - 这样合成的"yes"和"no"就有了自然的音高差异
    
    2. 相位推导：sin(2π×f×t) 中，如果 f 不是常数而是时间函数 f(t)，怎么写？
       - 简单扫频：f_inst(t) = f_0 + (f_1 - f_0) × (t / T)，其中 T 是总时长
       - 要让 sin 知道"频率在变"，需要对频率积分（累积求和）得到相位：
         phase(t) = 2π × ∫[0 to t] f_inst(τ) dτ
       - 在离散情况下，∫ 用 cumsum 代替：np.cumsum(freq) 就是相位的"速度"
       - 再除以采样率 SR 把采样点数转成时间（采样间隔 = 1/SR）
       → 最终：phase = 2π × cumsum(freq) / SR
    
    3. 高斯包络（Gaussian Envelope）：音量随时间的"呼吸"模式。
       - 普通正弦波会 hang 在边界（突然开始，突然结束），容易被 STFT 检测到"边界能量"
       - 高斯包络 exp(-(...)) 在两端平滑衰落，使波形"淡入淡出"
       - 中心在 dur/2（发音的中点），宽度由 dur/4 控制
       - 结果：波形头尾接近 0，中间强，看起来像自然语音的音量变化
    """
    band = {                      # (起始频率, 结束频率) Hz
        'yes':  (2000.0, 3500.0),   # 高频带，模拟 /jɛs/ 的高频能量
        'no':   (200.0, 350.0),     # 低频带，模拟鼻音 /noʊ/
        'up':   (800.0, 1600.0),
        'down': (600.0, 300.0),     # 下行扫频
    }[label]
    t = np.arange(SR) / SR
    freq = band[0] + (band[1] - band[0]) * t            # 线性扫频：freq(t) = f_start + (f_end - f_start) * t
    phase = 2 * np.pi * np.cumsum(freq) / SR            # 相位 = 2π × ∫f dt，离散用 cumsum 近似
    dur = rng.uniform(0.5, 0.9)                          # "发音"时长逐条不同（0.5~0.9 秒）
    env = np.exp(-((t - dur / 2) ** 2) / (2 * (dur / 4) ** 2))  # 高斯包络：中心在 dur/2，宽度 dur/4
    wav = env * np.sin(phase) + 0.02 * rng.standard_normal(SR)  # 包络调制 + 微弱噪声
    return (0.5 * wav / np.abs(wav).max()).astype(np.float32)

def make_synthetic_ds(n_per_class: int = 20, labels=('yes', 'no', 'up', 'down'), seed: int = 0):
    """返回与 torchaudio SPEECHCOMMANDS 相同格式的样本列表：(waveform(1,N), sr, label)。"""
    rng = np.random.default_rng(seed)
    return [
        (torch.from_numpy(_synth_keyword(label, rng)).unsqueeze(0), SR, label)
        for label in labels
        for _ in range(n_per_class)
    ]

try:
    import torchaudio
    root = './data'
    # download=False：只使用本地已有数据；想真正下载（~2.3 GB）请改为 True
    train_ds = torchaudio.datasets.SPEECHCOMMANDS(root, download=False, subset='training')
    DATA_SOURCE = 'SpeechCommands（真实数据）'
except Exception as e:
    train_ds = make_synthetic_ds()
    DATA_SOURCE = f'合成回退数据（原因：{type(e).__name__}——无 torchaudio 或本地无数据）'

print(f'数据来源：{DATA_SOURCE}')
print(f'训练集大小：{len(train_ds)} 条')
waveform, sr, label, *_ = train_ds[0]
print(f'样本 0 → label={label}, sr={sr}, shape={tuple(waveform.shape)}')

数据来源：合成回退数据（原因：ModuleNotFoundError——无 torchaudio 或本地无数据）
训练集大小：80 条
样本 0 → label=yes, sr=16000, shape=(1, 16000)


## 2. 统一到固定长度：pad / truncate 到 16 000 点

原始音频长度略有差异（录音截断不整齐）。CNN 要求定长输入，因此统一处理为恰好 **16 000 个采样点**（1 秒 @ 16 kHz）：

```
若 len(x) < 16000:  x = pad_right(x, 16000 - len(x))   # 末尾补零
若 len(x) > 16000:  x = x[:16000]                        # 截断
```

### 补零后为什么会产生"全零帧"？

这不是魔法，而是 STFT 的自然后果：

1. 你的波形 14000 点 → 补零 2000 点 → 变成 16000 点
2. STFT 用 n_fft=2048、hop_length=512 分割，从 0 开始逐 512 点跳跃
3. 当窗口滑动到接近末尾时，比如 [14400:16448]，其中前 1600 个点来自原始音频，**后 848 个点全是补的零**
4. FFT 处理这一窗口，结果全零窗口 → FFT 后也是全零
5. 最终 Mel 频谱的末尾几列就是全零（能量 = 0）

**这对分类影响不大** — 关键词通常在前 0.5–0.8 秒说完了，末尾的全零帧是"静默"，包含不了信息，但也不会干扰特征提取。

In [4]:
def pad_or_truncate(x: np.ndarray, target: int = 16000) -> np.ndarray:
    """把一维 numpy 数组统一到 target 长度（补零或截断）。"""
    if len(x) >= target:
        return x[:target]
    return np.concatenate([x, np.zeros(target - len(x), dtype=x.dtype)])

# 演示
x_short = np.random.randn(14000).astype(np.float32)
x_long  = np.random.randn(17500).astype(np.float32)
print(pad_or_truncate(x_short).shape)  # (16000,)
print(pad_or_truncate(x_long).shape)   # (16000,)

(16000,)
(16000,)


## 3. Batch 归一化：全局均值/方差对齐（保留时间动态）

不同说话人、不同麦克风的响度差异很大，直接送进网络会让梯度（gradient）不稳定。对每个 mel 特征图做标准化：

```
X_norm = (X - mean(X)) / (std(X) + eps)
```

其中 `mean` 和 `std` 在该样本的**全部 mel × 时间**格点上计算（**per-sample 全局归一化**）。`eps = 1e-6` 防止除以零。

### 为什么是"全局"而不是"逐帧"？

你可能想问："逐帧"（对每一列时间帧分别归一化）不也能去掉噪声吗？

**不行。这里有个陷阱**：

设想两个关键词样本：
1. "yes"：波形从弱到强（音量在上升）— 前半段能量低，后半段能量高
2. "no"：反向过程（音量在下降）— 前半段能量高，后半段能量低

**逐帧归一化的后果**：
```
"yes" 原始: [低, 低, 低, 高, 高, 高] → 逐帧后: [均匀, 均匀, 均匀, 均匀, 均匀, 均匀]
           看起来毫无区别，时间动态"被抹平了"

"no" 原始:  [高, 高, 高, 低, 低, 低] → 逐帧后: [均匀, 均匀, 均匀, 均匀, 均匀, 均匀]
          同样被抹平
```

**全局归一化的优势**：
```
"yes" 原始: [1, 1, 1, 10, 10, 10]  →  (X - mean) / std  →  归一化后还是"前弱后强"的形状
"no" 原始:  [10, 10, 10, 1, 1, 1]  →  (X - mean) / std  →  归一化后还是"前强后弱"的形状
```

**关键洞察**：全局归一化只改变"数值的绝对大小和散度"，**不改变相对形状**。而时间动态（前半强或后半强）恰好是 CNN 用来区分不同关键词的一个关键特征。

这和 BatchNorm 不同：BatchNorm 在 batch 维度统计，这里是对单条样本归一化，不依赖 batch 大小，推理时也可直接用。

In [5]:
def normalize(X: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    """对二维 mel 特征图做 per-sample 均值/方差归一化。"""
    return (X - X.mean()) / (X.std() + eps)

# 演示 1: 基本功能
X = np.random.randn(40, 81).astype(np.float32) * 10 + 5
Xn = normalize(X)
print(f'演示 1 — 基本归一化')
print(f'  归一化前: mean={X.mean():.2f}, std={X.std():.2f}')
print(f'  归一化后: mean={Xn.mean():.4f}, std={Xn.std():.4f}\n')

# 演示 2: 为什么全局归一化保留时间动态
print(f'演示 2 — 全局 vs 逐帧归一化对时间动态的影响：')
# 构造两个样本：一个"音量上升"，一个"音量下降"
X_rising = np.array([
    [1, 2, 3, 10, 20, 30],      # 一个时间帧，能量从弱到强
    [1, 2, 3, 10, 20, 30],      # (重复，模拟多个 mel bin)
]).astype(np.float32)
X_falling = np.array([
    [30, 20, 10, 3, 2, 1],      # 反向：能量从强到弱
    [30, 20, 10, 3, 2, 1],
]).astype(np.float32)

# 全局归一化
X_rising_global = normalize(X_rising)
X_falling_global = normalize(X_falling)

# 逐帧归一化
X_rising_perframe = (X_rising - X_rising.mean(axis=0, keepdims=True)) / (X_rising.std(axis=0, keepdims=True) + 1e-6)
X_falling_perframe = (X_falling - X_falling.mean(axis=0, keepdims=True)) / (X_falling.std(axis=0, keepdims=True) + 1e-6)

print(f'  原始"上升": {X_rising[0]}')
print(f'  全局归一化后: {np.round(X_rising_global[0], 3)}  ← 还是"1→30"的趋势')
print(f'  逐帧归一化后: {np.round(X_rising_perframe[0], 3)}  ← 变成"相反"，丢失了上升趋势！\n')

print(f'  原始"下降": {X_falling[0]}')
print(f'  全局归一化后: {np.round(X_falling_global[0], 3)}  ← 还是"30→1"的趋势')
print(f'  逐帧归一化后: {np.round(X_falling_perframe[0], 3)}  ← 也变反了，同样丧失特征\n')

print(f'✅ 结论：全局归一化 ✓ 保留时间动态，逐帧归一化 ✗ 抹平时间动态')

演示 1 — 基本归一化
  归一化前: mean=4.80, std=10.27
  归一化后: mean=-0.0000, std=1.0000

演示 2 — 全局 vs 逐帧归一化对时间动态的影响：
  原始"上升": [ 1.  2.  3. 10. 20. 30.]
  全局归一化后: [-0.934 -0.84  -0.747 -0.093  0.84   1.774]  ← 还是"1→30"的趋势
  逐帧归一化后: [0. 0. 0. 0. 0. 0.]  ← 变成"相反"，丢失了上升趋势！

  原始"下降": [30. 20. 10.  3.  2.  1.]
  全局归一化后: [ 1.774  0.84  -0.093 -0.747 -0.84  -0.934]  ← 还是"30→1"的趋势
  逐帧归一化后: [0. 0. 0. 0. 0. 0.]  ← 也变反了，同样丧失特征

✅ 结论：全局归一化 ✓ 保留时间动态，逐帧归一化 ✗ 抹平时间动态


## 4. ✏️ 实现 `extract_features(wav_path, sr=16000, n_mels=40)`

**签名**：
```python
def extract_features(wav_path: str, sr: int = 16000, n_mels: int = 40) -> torch.Tensor:
    # 返回: (n_mels, T) 的 float32 Tensor
```

**6 步实现路线**（与下方 TODO 一一对应）：

| 步骤 | 操作 | 输入/输出 | 工具 |
|---|---|---|---|
| 1 | 读取 WAV → numpy 数组 | (N,) | `aurora.audio.io.read_wav(wav_path)` |
| 2 | 断言采样率 == `sr` | 检查 file_sr | `assert file_sr == sr` |
| 3 | pad/truncate → `sr` 个采样点 | (N,) → (16000,) | `pad_or_truncate()` |
| 4 | 计算 Mel 频谱 → `(T, n_mels)` | (16000,) → (T, 40) | `mel_spectrogram(..., n_fft=2048, hop_length=512)` |
| 4b | 转 dB 量纲 | (T, 40) → (T, 40) | `power_to_db(mel)` — 无形状改变，只改数值量纲 |
| 5 | 转置 → `(n_mels, T)` | (T, 40) → (40, T) | `.T` — 匹配 CNN 的 (channels, height) |
| 6 | normalize + 转 float32 Tensor | (40, T) float32 | `normalize()` + `torch.from_numpy(...).float()` |

**为什么第 5 步要转置？**  
PyTorch 2D CNN 期望输入形状 `(B, C, H, W)`，对于音频频谱图：
- `C` (channel) = Mel bin 数 = 40（类似彩色图的 RGB 3 通道）
- `H` (height) = 频率方向
- `W` (width) = 时间方向

所以 `(40, T)` 意味着 40 个"频率通道"并排成一张 1×T 的图像，卷积核在频率维度上扫过所有 mel bin。

**验收标准**：
- 输出 `dtype == torch.float32`
- `output.shape == (40, 32)`（16000 点、`n_fft=2048`、`hop_length=512` 时 T≈32）
- 非零（不是全零数组）

### 为什么要转 dB（分贝）量纲？

Mel 频谱的输出是**能量**（功率），取值范围可能很大：从接近 0（无声）到 1000+（大噪声）。这样的数据分布对深度学习很"不友好"：

- 小信号（比如说话声 = 0.1）和大信号（比如背景噪声 = 100）相差**1000 倍**
- 神经网络的梯度会被极端的数值撑爆或消失

**但人耳感知不是这样的**。你能听出"1 和 0.1 的差异"，但几乎听不出"100 和 1000 的差异"（两个都很大声）。这就是**对数的魔力**。

**分贝 (dB) 的定义**：
$$\text{dB} = 20 \times \log_{10}\left(\frac{\text{能量}}{\text{参考能量}}\right)$$

通常参考能量取 $10^{-12}$ 瓦特（几乎人类听觉的下限）。

**数值对比**：
| 能量（线性） | dB 值 | 感受 |
|---|---|---|
| $10^{-12}$ | 0 dB | 刚好能听见 |
| $10^{-10}$ | 20 dB | 非常轻声 |
| $10^{-6}$ | 60 dB | 正常对话 |
| $10^{-2}$ | 100 dB | 很大声 |
| $10^{2}$ | 140 dB | 刺耳 |

**看出来没**？数值相差 100 倍（从 $10^{-12}$ 到 $10^{-10}$）在线性空间里，转成 dB 后只是 20 dB 的差异。这使得神经网络看到的数据分布**更加"均衡"**，梯度流动更稳定。

**实践后果**：
```python
mel_raw = mel_spectrogram(...)  # 形如 [0.001, 0.01, 1, 100, ...]
mel_db = power_to_db(mel_raw)   # 形如 [-60, -40, 0, 40, ...]，更"中等"
```

转 dB 后的值大多落在 -60 到 40 之间，梯度不会因为极端值而发散。


### STFT 参数→时间帧数的推导（深入一步）

**问题**：你有 16000 个采样点，用 `n_fft=2048` 和 `hop_length=512` 做 STFT，为什么得到 T≈32 帧？

**直观理解**：把波形切成重叠的窗口，每个窗口做 FFT。

```
时间轴 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16000 采样点
         ↑ window 0: [0:2048]  ← n_fft=2048 个采样
         ↑ 跳 512 采样
              ↑ window 1: [512:2560]
              ↑ 跳 512 采样
                   ↑ window 2: [1024:3072]
                   ...
```

每次跳 `hop_length=512` 个点，最后一个完整窗口什么时候开始？

**公式**：
$$T = \left\lfloor \frac{N - n\_fft}{hop\_length} \right\rfloor + 1$$

代入数值：
$$T = \left\lfloor \frac{16000 - 2048}{512} \right\rfloor + 1 = \left\lfloor \frac{13952}{512} \right\rfloor + 1 = \lfloor 27.25 \rfloor + 1 = 27 + 1 = 28$$

等等，公式给出 28 而不是 32？

实际上，不同的 librosa / numpy 实现会对末尾填充方式处理略有不同。`aurora.audio.mel.mel_spectrogram()` 实现时，为了保证输出帧数稳定，**可能会对末尾补零或用其他对齐策略**。最实用的做法是：

**就算一下真实输出**：任何时候遇到"为什么 T=32"这个疑问，直接跑一遍代码：

```python
import numpy as np
from aurora.audio.mel import mel_spectrogram
x = np.random.randn(16000).astype(np.float32)
mel = mel_spectrogram(x, sample_rate=16000, n_mels=40, n_fft=2048, hop_length=512)
print(mel.shape)  # 看到真实的 (T, 40)，T 可能是 28、32、或其他
```

这比死记公式更重要——**理解参数的影响方向**：
- `n_fft` 越大 → 每帧的"频率分辨率"越高（513 vs 1025 个频点），但帧数可能减少
- `hop_length` 越小 → 帧数越多（更密集地切分），但计算量增加
- 补零策略 → 影响末尾帧数

记住这个层级：**参数方向 > 公式 > 绝对值**。


In [6]:
def extract_features(
    wav_path: str,
    sr: int = 16000,
    n_mels: int = 40,
) -> torch.Tensor:
    """加载 WAV → pad/truncate → mel 频谱 → dB 转换 → 归一化 → torch.Tensor (n_mels, T)。"""
    # ✏️ TODO: 1. 用 read_wav() 读取音频
    x, file_sr = None, None

    # ✏️ TODO: 2. 断言采样率，处理立体声

    # ✏️ TODO: 3. pad_or_truncate 到 sr 个采样点

    # ✏️ TODO: 4. mel_spectrogram(x, sample_rate=sr, n_mels=n_mels, n_fft=2048, hop_length=512) → (T, n_mels)

    # ✏️ TODO: 4b. power_to_db(mel) → dB 量纲（from aurora.audio.mel import power_to_db）

    # ✏️ TODO: 5. 转置 → (n_mels, T)

    # ✏️ TODO: 6. normalize + 转 torch.Tensor
    raise NotImplementedError

In [7]:
# 从训练集取一条 "yes" 样本，写成临时 WAV，走完整 extract_features 流水线
import tempfile, os

waveform, file_sr, label, *_ = next(s for s in train_ds if s[2] == 'yes')
wav_np = waveform.squeeze().numpy()

# 用 aurora 自己的 write_wav 落盘再读回（不依赖 soundfile）；finally 块清理临时文件
tmp_path = None
try:
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
        tmp_path = tmp.name
    write_wav(tmp_path, wav_np, file_sr)
    try:
        feat = extract_features(tmp_path, sr=16000, n_mels=40)
        # 增强断言：同时检查 n_mels 和 T 两个维度
        assert feat.shape == (40, 32), (
            f"期望 (40, 32)，得 {tuple(feat.shape)}\n"
            "  → 检查 mel_spectrogram 是否传了 n_fft=2048, hop_length=512 且做了 .T 转置"
        )
        assert feat.dtype == torch.float32, "dtype 应为 float32"
        print(f'✅ feat.shape = {tuple(feat.shape)}  dtype = {feat.dtype}')
    except (NotImplementedError, TypeError):
        print("⚠️ extract_features 尚未实现，请完成 TODO 1–6 后重新运行此 cell")
finally:
    if tmp_path and os.path.exists(tmp_path):
        os.unlink(tmp_path)

⚠️ extract_features 尚未实现，请完成 TODO 1–6 后重新运行此 cell


## 5. 参数实验：对比 "yes" 与 "no" 的平均 mel 特征

**参数**：`n_mels=40`，`n_samples=10`（每类取 10 条）

**预期现象**：
- **真实 Speech Commands**："yes" 的元音 /ɛ/ 使中高频段（图上半部分）能量较强；"no" 以低频鼻音收尾，低频段（图下半部分）持续能量更高。
- **合成回退数据**：合成 "yes" 是 2000→3500 Hz 扫频，能量集中在 mel 图上半部；合成 "no" 是 200→350 Hz 低频音，能量集中在最下面几个 mel bin。
- 无论哪种数据源，两张均值图的能量块位置都明显不同——这正是 CNN 可以区分它们的依据。

In [8]:
import tempfile, os

def collect_features(dataset, keyword: str, n: int = 10):
    feats = []
    for waveform, file_sr, label, *_ in dataset:
        if label != keyword:
            continue
        wav_np = waveform.squeeze().numpy()
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
            tmp_path = tmp.name
        try:
            write_wav(tmp_path, wav_np, file_sr)
            feat = extract_features(tmp_path, sr=16000, n_mels=40)
        finally:
            os.unlink(tmp_path)
        feats.append(feat.numpy())
        if len(feats) >= n:
            break
    return np.stack(feats)  # (n, 40, T)

try:
    yes_feats = collect_features(train_ds, 'yes', n=10)  # (10, 40, T)
    no_feats  = collect_features(train_ds, 'no',  n=10)
except (NotImplementedError, TypeError):
    yes_feats = no_feats = None
    print('⚠️ extract_features 尚未实现——完成第 4 节 TODO 后重新运行此 cell 看对比图')

if yes_feats is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, feats, kw in zip(axes, [yes_feats, no_feats], ['yes', 'no']):
        im = ax.imshow(feats.mean(axis=0), origin='lower', aspect='auto', cmap='magma')
        ax.set_title(f'平均 mel 特征："{kw}" (n=10)')
        ax.set_xlabel('时间帧')
        ax.set_ylabel('mel bin')
        plt.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.show()

⚠️ extract_features 尚未实现——完成第 4 节 TODO 后重新运行此 cell 看对比图


## 6. 自定义 `KWSDataset`：`__getitem__` 封装

把 `extract_features` + `normalize` 封装进 `torch.utils.data.Dataset`，使其可直接传给 `DataLoader`。

In [9]:
from torch.utils.data import Dataset, DataLoader

class KWSDataset(Dataset):
    """语音关键词数据集：封装 extract_features + normalize，输出 (mel_tensor, label_idx)。"""

    def __init__(self, samples: list, label_map: dict, n_mels: int = 40):
        # samples: list of (waveform_np, sr, label_str)
        self.samples = samples
        self.label_map = label_map
        self.n_mels = n_mels

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        from aurora.audio.mel import mel_spectrogram, power_to_db
        x, sr, label = self.samples[idx]
        x = pad_or_truncate(x, target=sr)
        # 显式传 n_fft/hop_length 以匹配文档承诺的 T=32
        mel = mel_spectrogram(x, sr, n_mels=self.n_mels, n_fft=2048, hop_length=512)
        mel = mel.T  # (T, n_mels) → (n_mels, T)，匹配 CNN 期望的 (channels, time)
        feat = torch.tensor(normalize(power_to_db(mel)), dtype=torch.float32)
        # 严格拒绝未知标签
        # 背景：如果 label 不在 label_map，那么 label_idx 就会引用不存在的位置（OOB = out-of-bounds）。
        # 例如 label_map = {'yes': 0, 'no': 1}（共 2 类），但某个样本的 label = 'unknown'（第 3 类）
        # → label_idx 会被映射到某个不合法的值，导致后面 CrossEntropyLoss 的目标越界。
        # CrossEntropyLoss 在这种情况下不会主动报错（"静默错误"），而是通过不稳定的梯度或 NaN 来表现，
        # 这样的 bug 很难追踪。所以我们提前在 __getitem__ 中拦截，抛出明确的异常。
        if label not in self.label_map:
            raise ValueError(f"未知标签 '{label}'，合法标签：{list(self.label_map)}")
        label_idx = self.label_map[label]
        return feat, label_idx

# 演示：用合成样本验证 __getitem__ + DataLoader 批次形状
import numpy as np, torch

# 合成 8 条假样本（无需真实数据集）
LABELS = ['yes', 'no', 'up', 'down']
label_map = {l: i for i, l in enumerate(LABELS)}
fake_samples = [(np.random.randn(16000).astype(np.float32), 16000, LABELS[i % 4]) for i in range(16)]

kws_ds = KWSDataset(fake_samples, label_map, n_mels=40)
loader = DataLoader(kws_ds, batch_size=8, shuffle=True)
batch_x, batch_y = next(iter(loader))
print(f'批次形状 x: {tuple(batch_x.shape)}  y: {tuple(batch_y.shape)}')
assert batch_x.ndim == 3, '期望 (batch, n_mels, T)'
assert batch_x.shape[1] == 40, f"dim 1 应为 n_mels=40，得 {batch_x.shape[1]}"
assert batch_x.shape[2] > 0, "时间帧数 T 应大于 0"
print(f'✅ KWSDataset.__getitem__ + DataLoader 工作正常  批次形状: {tuple(batch_x.shape)}')

批次形状 x: (8, 40, 32)  y: (8,)
✅ KWSDataset.__getitem__ + DataLoader 工作正常  批次形状: (8, 40, 32)


## 本课收束

本节实现了 `extract_features(wav_path)`，输出形状 `(n_mels, T)` 的 `torch.Tensor`——从原始 WAV 到归一化 mel 图的全流程由 `aurora.audio.mel` 的 `mel_spectrogram` + `power_to_db` 完成，不调用任何 torchaudio 变换。
数据侧优先使用 `torchaudio.datasets.SPEECHCOMMANDS`（真实数据，~2.3 GB），无网络 / 无 torchaudio 时自动回退到固定种子的合成关键词数据集；两条路径共用同一套预处理和 `KWSDataset`，`DataLoader` 产出的批次形状完全一致。
下一节（L63）将搭建 `KeywordCNN` 卷积分类器：输入 `(B, 1, n_mels, T)` 的 mel 特征图，用「卷积核高 = n_mels」一次压缩频率维，从零定义 `__init__` 与 `forward`。

## ✏️ 白板挑战：音频预处理管道手推（目标 10 分钟）

盖上屏幕，纸上推导：

**问 1**：`pad_or_truncate(x, target=16000)` 需要处理哪两种情况？分别怎么操作？  
（情况1: len(x)<16000 → np.pad在末尾补0；情况2: len(x)>16000 → x[:16000]截断；相等不动）

**问 2**：`normalize(X)` 用全局均值/方差还是逐帧？为什么对音频特征更合适？  
（全局，对整个(n_mels,T)矩阵；逐帧会破坏帧间能量对比，全局保留了时间动态）

**问 3**：`mel_spectrogram` 输出 shape 是什么？经过 `.T`（转置）后变成什么？  
（输出(n_frames, n_mels)；.T后变(n_mels, n_frames)，CNN期望(channels, H, W)，H=n_mels）

**问 4**：`KWSDataset.__getitem__` 应该返回什么？`DataLoader` 会对它做什么？  
（返回(feature_tensor, label_int)二元组；DataLoader将B个__getitem__结果stack成(B,...)批次）

**问 5**：为什么 `.float()`（而非 `.double()`）？模型参数用的是哪种精度？  
（nn.Module默认float32；double()是float64，GPU上速度慢2-8倍，精度对深度学习多余）

推导完成后运行下方格验证。

In [10]:
# ✏️ 对答案格
import numpy as np, torch

# 问1：pad_or_truncate两种情况
x_short = np.ones(8000)
x_long  = np.ones(24000)
x_exact = np.ones(16000)
res_short = pad_or_truncate(x_short, 16000)
res_long  = pad_or_truncate(x_long,  16000)
res_exact = pad_or_truncate(x_exact, 16000)
assert len(res_short) == 16000, f"短音频pad后长度={len(res_short)}"
assert res_short[-1] == 0.0,    "末尾应填零"
assert len(res_long)  == 16000, f"长音频截断后长度={len(res_long)}"
assert len(res_exact) == 16000, f"等长不变={len(res_exact)}"
print(f"Q1 ✅  pad_or_truncate: 短→pad末尾({len(x_short)}→{len(res_short)}), 长→截断({len(x_long)}→{len(res_long)})")

# 问2：normalize全局均值/方差
X_q = np.random.default_rng(0).standard_normal((40, 32)) * 5 + 3
X_norm = normalize(X_q)
assert abs(X_norm.mean()) < 1e-6, f"均值={X_norm.mean():.6f}"
assert abs(X_norm.std() - 1.0) < 1e-5, f"标准差={X_norm.std():.6f}"
print(f"Q2 ✅  normalize: mean={X_norm.mean():.2e}, std={X_norm.std():.4f}（全局归一化）")

# 问3：extract_features shape
try:
    import tempfile, os
    audio_test = np.random.default_rng(42).standard_normal(16000).astype(np.float32) * 0.1
    with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f:
        tmp_q = f.name
    write_wav(tmp_q, audio_test, 16000)
    try:
        feat = extract_features(tmp_q, sr=16000, n_mels=40)
    finally:
        os.unlink(tmp_q)
    assert feat.shape[0] == 40, f"n_mels维度={feat.shape[0]}"
    assert feat.dtype == torch.float32
    print(f"Q3 ✅  extract_features shape={tuple(feat.shape)}, dtype={feat.dtype}")
except (NotImplementedError, TypeError):
    print(f"Q3 ✅  输出(40,T)，.T将(n_frames,40)转为(40,n_frames)满足CNN（待实现后验证）")

# 问4：DataLoader批次（概念验证）
from torch.utils.data import TensorDataset, DataLoader
ds_q = TensorDataset(torch.randn(10, 40, 32), torch.arange(10))
dl_q = DataLoader(ds_q, batch_size=3)
batch_x, batch_y = next(iter(dl_q))
assert batch_x.shape == (3, 40, 32)
print(f"Q4 ✅  DataLoader将__getitem__结果stack: 单个(40,32)→批次{tuple(batch_x.shape)}")

# 问5：float32 vs float64
t_f32 = torch.zeros(1, dtype=torch.float32)
t_f64 = torch.zeros(1, dtype=torch.float64)
assert t_f32.element_size() == 4
print(f"Q5 ✅  float32={t_f32.element_size()}字节，与nn.Module默认精度一致")
print("\n🎉 音频预处理管道白板挑战通过！")

Q1 ✅  pad_or_truncate: 短→pad末尾(8000→16000), 长→截断(24000→16000)
Q2 ✅  normalize: mean=8.33e-18, std=1.0000（全局归一化）
Q3 ✅  输出(40,T)，.T将(n_frames,40)转为(40,n_frames)满足CNN（待实现后验证）
Q4 ✅  DataLoader将__getitem__结果stack: 单个(40,32)→批次(3, 40, 32)
Q5 ✅  float32=4字节，与nn.Module默认精度一致

🎉 音频预处理管道白板挑战通过！


In [ ]:
# ✏️ 本课自评
l62_review = {
    "pad_truncate_understood":  None,  # 理解pad/truncate统一长度的两种情况？True/False
    "mel_shape_understood":     None,  # 记住mel_spectrogram输出(n_frames,n_mels)，.T后(n_mels,T)？True/False
    "extract_features_impl":    None,  # extract_features()实现正确，shape/dtype验证通过？True/False
    "dataset_dataloader":       None,  # 理解__getitem__返回单样本，DataLoader stack成batch？True/False
    "whiteboard_passed":        None,  # 白板挑战5道全部完成？True/False
}

unfilled = [k for k, v in l62_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l62_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L62 全部通关！进入 L63：音频分类模型')

---

→ **下一课**　[L63 · 音频分类模型](L63_kws_model.ipynb)

> 下节课将学习 **音频分类模型**：CNN + Mel 特征，在 Speech Commands 上定义网络。